In [1]:
# ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# تحميل نموذج Haar Cascade للوجوه
import cv2

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


In [11]:
# استيراد المكتبات
import os
import glob
import cv2
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# تحديد مسارات مجلدات البيانات والإخراج
DATASET_DIR = '/content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2'
CSV_OUTPUT_PATH = '/content/drive/MyDrive/celeb_df_dataset.csv'



In [4]:
# قراءة أسماء فيديوهات الاختبار من الملف النصي
test_list_path = os.path.join(DATASET_DIR, 'List_of_testing_videos.txt')

# تأكد من وجود الملف قبل القراءة
if not os.path.exists(test_list_path):
    raise FileNotFoundError(f"File not found: {test_list_path}")

# قراءة أسماء الفيديوهات
with open(test_list_path, 'r') as f:
    test_videos = set([line.strip() for line in f.readlines()])

# طباعة عدد فيديوهات الاختبار
print(f'Number of test videos: {len(test_videos)}')


Number of test videos: 518


In [5]:
def frame_extract(path):
    vidObj = cv2.VideoCapture(path)
    success = True
    while success:
        success, image = vidObj.read()
        if success:
            yield image


In [6]:
!pip install mediapipe opencv-python


In [7]:
import mediapipe as mp
import cv2
import os
import pandas as pd

def process_videos_mediapipe(input_dir, output_dir, label, csv_path):
    mp_face_detection = mp.solutions.face_detection
    face_detection = mp_face_detection.FaceDetection(
        model_selection=1,
        min_detection_confidence=0.3
    )

    os.makedirs(output_dir, exist_ok=True)
    face_data = []
    failed_videos = []

    for video_file in os.listdir(input_dir):
        if not video_file.endswith(".mp4"):
            continue

        video_path = os.path.join(input_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        face_written = 0
        output_path = os.path.join(output_dir, video_file)
        writer = None

        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            frame_count += 1
            if frame_count % 5 != 0:
                continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_detection.process(frame_rgb)

            if results.detections:
                for detection in results.detections:
                    bbox = detection.location_data.relative_bounding_box
                    h, w, _ = frame.shape
                    x = int(bbox.xmin * w)
                    y = int(bbox.ymin * h)
                    width = int(bbox.width * w)
                    height = int(bbox.height * h)

                    x = max(x, 0)
                    y = max(y, 0)
                    width = min(width, w - x)
                    height = min(height, h - y)

                    face = frame[y:y + height, x:x + width]
                    face = cv2.resize(face, (112, 112))

                    if writer is None:
                        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                        writer = cv2.VideoWriter(output_path, fourcc, 5.0, (112, 112))

                    writer.write(face)
                    face_written += 1

            if face_written >= 20:
                break

        cap.release()
        if writer is not None:
            writer.release()

        if face_written > 0:
            face_data.append((video_file, label))
        else:
            if os.path.exists(output_path):
                os.remove(output_path)
            failed_videos.append(video_file)

    # حفظ في CSV (add to it)
    df = pd.DataFrame(face_data, columns=['file', 'label'])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

    # حفظ الفيديوهات الفاشلة
    failed_txt_path = os.path.join(output_dir, f"failed_{os.path.basename(input_dir)}_{label}.txt")
    with open(failed_txt_path, 'w') as f:
        for video_name in failed_videos:
            f.write(video_name + "\n")

    print(f"✅ Processed {len(face_data)} videos from {input_dir}")
    print(f"❌ Failed {len(failed_videos)} videos from {input_dir}")


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


In [8]:
from tqdm import tqdm
import pandas as pd

# مسار CSV الموحد
csv_path = '/content/drive/MyDrive/Gobal_metadata.csv'

# إزالة ملف CSV السابق إذا كنت تريد إعادة الكتابة من الصفر
# (اختياري)
#if os.path.exists(csv_path):
#    os.remove(csv_path)

# قائمة المجلدات
calls = [
    {
        'input_dir': '/content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-real',
        'output_dir': '/content/drive/MyDrive/Celeb_face_only/Celeb-real',
        'label': 'REAL'
    },
    {
        'input_dir': '/content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/YouTube-real',
        'output_dir': '/content/drive/MyDrive/Celeb_face_only/YouTube-real',
        'label': 'REAL'
    },
    {
        'input_dir': '/content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-synthesis',
        'output_dir': '/content/drive/MyDrive/Celeb_face_only/Celeb-synthesis',
        'label': 'FAKE'
    }
]

print("🚀 Starting batch processing...\n")
for call in tqdm(calls, desc="📁 Processing folders", unit="folder"):
    process_videos_mediapipe(
        input_dir=call['input_dir'],
        output_dir=call['output_dir'],
        label=call['label'],
        csv_path=csv_path
    )

print(f"\n📄 All face video data saved to: {csv_path}")


🚀 Starting batch processing...



📁 Processing folders:  33%|███▎      | 1/3 [04:26<08:53, 266.92s/folder]

✅ Processed 589 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-real
❌ Failed 1 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-real


📁 Processing folders:  67%|██████▋   | 2/3 [06:24<02:58, 178.80s/folder]

✅ Processed 300 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/YouTube-real
❌ Failed 0 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/YouTube-real


📁 Processing folders: 100%|██████████| 3/3 [39:35<00:00, 791.90s/folder] 

✅ Processed 5648 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-synthesis
❌ Failed 0 videos from /content/drive/MyDrive/project/DATA_DEEP_FAKE/Celeb-DF-v2/Celeb-synthesis

📄 All face video data saved to: /content/drive/MyDrive/Gobal_metadata.csv


In [9]:
import os
import shutil
from pathlib import Path

# المسار الرئيسي الذي يحتوي على المجلدات الثلاثة
main_folder = "/content/drive/MyDrive/Celeb_face_only"  # غير هذا المسار حسب موقع مجلدك

# المجلد الجديد الذي سيحتوي على جميع الفيديوهات
output_folder = "/content/drive/MyDrive/all_videos_face"

# صيغ الفيديو المدعومة
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv', '.webm', '.m4v', '.mpeg', '.mpg')

def collect_videos(source_path, destination_path):
    """
    دالة لجمع جميع الفيديوهات من المجلدات الفرعية
    """
    # إنشاء المجلد الوجهة إذا لم يكن موجوداً
    os.makedirs(destination_path, exist_ok=True)

    video_count = 0

    # البحث عن جميع الملفات في المجلد الرئيسي والمجلدات الفرعية
    for root, dirs, files in os.walk(source_path):
        for file in files:
            # التحقق من أن الملف فيديو
            if file.lower().endswith(video_extensions):
                source_file = os.path.join(root, file)

                # إنشاء اسم فريد للملف في حال وجود ملفات بنفس الاسم
                destination_file = os.path.join(destination_path, file)

                # إذا كان الملف موجوداً، إضافة رقم للاسم
                if os.path.exists(destination_file):
                    name, ext = os.path.splitext(file)
                    counter = 1
                    while os.path.exists(destination_file):
                        destination_file = os.path.join(destination_path, f"{name}_{counter}{ext}")
                        counter += 1

                # نسخ الفيديو إلى المجلد الجديد
                shutil.copy2(source_file, destination_file)
                video_count += 1
                print(f"تم نسخ: {file}")

    return video_count

# تشغيل الدالة
print(f"جاري البحث عن الفيديوهات في: {main_folder}")
print("-" * 50)

total_videos = collect_videos(main_folder, output_folder)

print("-" * 50)
print(f"✓ تم الانتهاء! تم نسخ {total_videos} فيديو إلى المجلد: {output_folder}")

# عرض قائمة بالفيديوهات المنسوخة
print("\nقائمة الفيديوهات:")
for video in sorted(os.listdir(output_folder)):
    if video.lower().endswith(video_extensions):
        size = os.path.getsize(os.path.join(output_folder, video)) / (1024 * 1024)  # بالميجابايت
        print(f"  - {video} ({size:.2f} MB)")

Streaming output truncated to the last 5000 lines.
  - id21_id33_0008.mp4 (0.01 MB)
  - id21_id33_0009.mp4 (0.01 MB)
  - id21_id34_0000.mp4 (0.01 MB)
  - id21_id34_0001.mp4 (0.01 MB)
  - id21_id34_0002.mp4 (0.01 MB)
  - id21_id34_0004.mp4 (0.01 MB)
  - id21_id34_0005.mp4 (0.01 MB)
  - id21_id34_0006.mp4 (0.02 MB)
  - id21_id34_0007.mp4 (0.01 MB)
  - id21_id34_0009.mp4 (0.01 MB)
  - id21_id35_0000.mp4 (0.01 MB)
  - id21_id35_0001.mp4 (0.01 MB)
  - id21_id35_0002.mp4 (0.01 MB)
  - id21_id35_0004.mp4 (0.01 MB)
  - id21_id35_0005.mp4 (0.01 MB)
  - id21_id35_0006.mp4 (0.02 MB)
  - id21_id35_0007.mp4 (0.01 MB)
  - id21_id35_0008.mp4 (0.01 MB)
  - id21_id35_0009.mp4 (0.01 MB)
  - id21_id37_0000.mp4 (0.01 MB)
  - id21_id37_0001.mp4 (0.01 MB)
  - id21_id37_0002.mp4 (0.01 MB)
  - id21_id37_0004.mp4 (0.01 MB)
  - id21_id37_0005.mp4 (0.01 MB)
  - id21_id37_0006.mp4 (0.02 MB)
  - id21_id37_0007.mp4 (0.01 MB)
  - id21_id37_0008.mp4 (0.01 MB)
  - id21_id37_0009.mp4 (0.01 MB)
  - id21_id38_0000.mp4 (0

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
